In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

import folium
from geopy.geocoders import Nominatim
import time
import matplotlib
import matplotlib.cm as cm
from pathlib import Path

from matplotlib import font_manager, rc
import os

# Windows 기본 한글 폰트 경로
font_path = "C:/Windows/Fonts/malgun.ttf"

font = font_manager.FontProperties(fname=font_path).get_name()
rc("font", family=font)

# 마이너스 깨짐 방지
plt.rcParams["axes.unicode_minus"] = False

# 데이터(all_data_v1_update) 로드

In [7]:
# 현재 노트북 위치
BASE_DIR = Path().resolve()

# 프로젝트 루트 (notebooks의 상위 폴더)
PROJECT_ROOT = BASE_DIR.parent

# data 폴더
DATA_DIR = PROJECT_ROOT / "data"

# data 파일
path1 = DATA_DIR / "all_data_v1_update.csv"

# data load
df = pd.read_csv(path1, encoding="utf-8")

# 분기별로 데이터 압축

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85969 entries, 0 to 85968
Data columns (total 36 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   가맹점구분번호            85969 non-null  object 
 1   가맹점주소              85969 non-null  object 
 2   업종                 85969 non-null  object 
 3   상권                 85969 non-null  object 
 4   개설일                85969 non-null  object 
 5   폐업일                1761 non-null   object 
 6   폐업여부               85969 non-null  bool   
 7   기준년월               85969 non-null  object 
 8   가맹점 운영개월수 구간       85969 non-null  int64  
 9   매출금액 구간            85969 non-null  int64  
 10  매출건수 구간            85969 non-null  int64  
 11  유니크 고객 수 구간        85969 non-null  int64  
 12  객단가 구간             85969 non-null  int64  
 13  취소율 구간             85969 non-null  float64
 14  배달매출금액 비율          85969 non-null  float64
 15  동일 업종 매출금액 비율      85969 non-null  float64
 16  동일 업종 매출건수 비율      859

In [ ]:
# 분기 컬럼 생성
df["기준년월"] = pd.to_datetime(df["기준년월"])
df["연도"] = df["기준년월"].dt.year
df["분기"] = df["기준년월"].dt.quarter

In [17]:
df.columns

Index(['가맹점구분번호', '가맹점주소', '업종', '상권', '개설일', '폐업일', '폐업여부', '기준년월',
       '가맹점 운영개월수 구간', '매출금액 구간', '매출건수 구간', '유니크 고객 수 구간', '객단가 구간', '취소율 구간',
       '배달매출금액 비율', '동일 업종 매출금액 비율', '동일 업종 매출건수 비율', '동일 업종 내 매출 순위 비율',
       '동일 상권 내 매출 순위 비율', '동일 업종 내 해지 가맹점 비중', '동일 상권 내 해지 가맹점 비중',
       '남성 20대이하 고객 비중', '남성 30대 고객 비중', '남성 40대 고객 비중', '남성 50대 고객 비중',
       '남성 60대이상 고객 비중', '여성 20대이하 고객 비중', '여성 30대 고객 비중', '여성 40대 고객 비중',
       '여성 50대 고객 비중', '여성 60대이상 고객 비중', '재방문 고객 비중', '신규 고객 비중',
       '거주 이용 고객 비율', '직장 이용 고객 비율', '유동인구 이용 고객 비율', '연도', '분기'],
      dtype='object')

# 일부 컬럼만

### 배달여부 컬럼 생성

In [31]:
df["배달여부"] = (
    df.groupby("가맹점구분번호")["배달매출금액 비율"]
      .transform(lambda x: (x != 0).any())
      .astype(int)
)

In [33]:
# 지정한 컬럼
target_cols = [
    "가맹점 운영개월수 구간",
    "매출금액 구간",
    "유니크 고객 수 구간",
    "취소율 구간",
    "배달매출금액 비율",
    "배달여부",
    "재방문 고객 비중",
    '폐업여부',
]

# 상점×분기 압축 (평균/표준편차)
summary_df = (
    df.groupby(["가맹점구분번호", "연도", "분기"], dropna=False)[target_cols]
      .agg(["mean", "std"])
)

In [34]:
# 컬럼명 깔끔하게: (컬럼, mean/std) → "컬럼_mean" 형태
summary_df.columns = [f"{col}_{stat}" for col, stat in summary_df.columns]
summary_df = summary_df.reset_index()

# 선택: 표준편차가 NaN(분기 데이터 1개뿐이면 std가 NaN)인 경우 0으로
std_cols = [c for c in summary_df.columns if c.endswith("_std")]
summary_df[std_cols] = summary_df[std_cols].fillna(0)

summary_df.head()

,가맹점구분번호,연도,분기,가맹점 운영개월수 구간_mean,가맹점 운영개월수 구간_std,매출금액 구간_mean,매출금액 구간_std,유니크 고객 수 구간_mean,유니크 고객 수 구간_std,취소율 구간_mean,취소율 구간_std,배달매출금액 비율_mean,배달매출금액 비율_std,배달여부_mean,배달여부_std,재방문 고객 비중_mean,재방문 고객 비중_std,폐업여부_mean,폐업여부_std
0,000F03E44A,2023,1,5.000000,0.00000,6.000000,0.00000,5.666667,0.57735,1.208333,0.180422,69.283333,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,000F03E44A,2023,2,5.000000,0.00000,6.000000,0.00000,5.000000,0.00000,1.000000,0.000000,69.283333,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,000F03E44A,2023,3,5.000000,0.00000,6.000000,0.00000,5.666667,0.57735,1.208333,0.180422,69.283333,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,000F03E44A,2023,4,4.333333,0.57735,6.000000,0.00000,5.666667,0.57735,1.208333,0.180422,69.283333,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,000F03E44A,2024,1,4.000000,0.00000,5.666667,0.57735,5.333333,0.57735,1.104167,0.180422,69.283333,0.0,1.0,0.0,0.0,0.0,0.0,0.0


# 폐업 여부에 유의미하게 영향을 준 변수

In [35]:
# 폐업 여부 기준 분리
df_open = summary_df[summary_df["폐업여부_mean"] == 0]
df_closed = summary_df[summary_df["폐업여부_mean"] == 1]

In [36]:
test_vars = [
    "가맹점 운영개월수 구간_mean",
    "매출금액 구간_mean",
    "유니크 고객 수 구간_mean",
    "취소율 구간_mean",
    "배달매출금액 비율_mean",
    "배달여부_mean",
    "재방문 고객 비중_mean",
]


In [37]:
# Mann–Whitney U test
from scipy.stats import mannwhitneyu

results = []

for var in test_vars:
    stat, p = mannwhitneyu(
        df_open[var],
        df_closed[var],
        alternative="two-sided"
    )
    
    results.append({
        "변수": var,
        "U_statistic": stat,
        "p_value": p
    })

test_result_df = pd.DataFrame(results).sort_values("p_value")
test_result_df


,변수,U_statistic,p_value
0,가맹점 운영개월수 구간_mean,7025732.5,1.458074e-13
4,배달매출금액 비율_mean,7479557.0,3.105232e-09
5,배달여부_mean,7707459.5,2.805575e-06
6,재방문 고객 비중_mean,8772003.0,1.792646e-01
1,매출금액 구간_mean,8267545.5,2.500245e-01
3,취소율 구간_mean,8690011.0,3.274980e-01
2,유니크 고객 수 구간_mean,8676057.0,3.809041e-01


In [ ]:
# Wilcoxon rank-sum test
from scipy.stats import ranksums
import pandas as pd

results = []

for var in test_vars:
    stat, p = ranksums(
        df_open[var],
        df_closed[var]
    )
    
    results.append({
        "변수": var,
        "Z_statistic": stat,   # ranksums는 Z 통계량
        "p_value": p
    })

test_result_df = pd.DataFrame(results).sort_values("p_value")
test_result_df


,변수,Z_statistic,p_value
0,가맹점 운영개월수 구간_mean,-7.268769,3.627779e-13
4,배달매출금액 비율_mean,-5.030724,4.886315e-07
5,재방문 고객 비중_mean,1.343001,1.792716e-01
1,매출금액 구간_mean,-1.144742,2.523162e-01
3,취소율 구간_mean,0.938656,3.479074e-01
2,유니크 고객 수 구간_mean,0.869841,3.843870e-01


In [ ]:
# Welch t-test
from scipy.stats import ttest_ind
import pandas as pd

results = []

for var in test_vars:
    stat, p = ttest_ind(
        df_open[var],
        df_closed[var],
        equal_var=False,      # ⭐ Welch's t-test 핵심
        nan_policy="omit"     # 결측치 안전 처리
    )
    
    results.append({
        "변수": var,
        "t_statistic": stat,
        "p_value": p
    })

test_result_df = pd.DataFrame(results).sort_values("p_value")
test_result_df


,변수,t_statistic,p_value
0,가맹점 운영개월수 구간_mean,-9.368325,1.262503e-19
4,배달매출금액 비율_mean,-4.402928,1.258829e-05
5,재방문 고객 비중_mean,2.018433,4.397323e-02
1,매출금액 구간_mean,-1.156363,2.479783e-01
2,유니크 고객 수 구간_mean,0.952496,3.412179e-01
3,취소율 구간_mean,0.651965,5.146648e-01
